In [10]:
import os

In [11]:
%pwd

'/Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization'

In [12]:
os.chdir("../")

In [13]:
%pwd

'/Users/akashkumarsinha/Desktop/Text_summerization'

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvoluationConfig:
  root_dir: Path
  data_path: Path
  model_path: Path
  tokenizer_path: Path
  matric_file_name: Path

In [15]:
from TextSummarizer.utils.common import read_yaml, create_directories
from TextSummarizer.constants import *

class configurationManager:
  def __init__(self, 
               config_file_path = CONFIG_FILE_PATH, 
               params_file_path = PARAMS_FILE_PATH):
    
    self.config = read_yaml(config_file_path)
    self.params = read_yaml(params_file_path)
    create_directories([self.config.artifacts_root])
  
  
  def get_model_evoluation_config(self) -> ModelEvoluationConfig:
    
    config = self.config.model_evoluation
    create_directories([config.root_dir])

    model_evoluation_config = ModelEvoluationConfig(
        root_dir = config.root_dir,
        data_path = config.data_path,
        model_path = config.model_path,
        tokenizer_path = config.tokenizer_path,
        matric_file_name = config.matric_file_name
    )

    return model_evoluation_config

In [16]:
# install the modern evaluation library (datasets.load_metric is deprecated)
%pip install -q evaluate

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset, load_from_disk
from streamlit import metric
from zmq import device
import evaluate
import torch
import pandas as pd


Note: you may need to restart the kernel to use updated packages.


In [17]:


class ModelEvoluation:
    def __init__(self, config: ModelEvoluationConfig):
        self.config = config
    # Evaluation

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(
    self,
    dataset,
    metric,
    model,
    tokenizer,
    column_text="dialogue",
    column_summary="summary",
    batch_size=16,
    device=None
):
        # Determine device if not provided
        if device is None:
            device = "mps" if (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()) else ("cuda" if torch.cuda.is_available() else "cpu")

        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size
                                                                ))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        from tqdm import tqdm

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)
        ):

            inputs = tokenizer(article_batch, max_length=256,truncation=True,
                        padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                          attention_mask= inputs["attention_mask"].to(device),
                          length_penalty=0.8, num_beams=8, max_length=128)
            ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''
            # finally ,we decode the  generated texts,
            # replace the token and add the decoded texts with the referance to the matric.
            decoded_summaries= [tokenizer.decode(s,skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
                                for s in summaries]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)
# finally compute and return the rouge scores


        score = metric.compute()
        return score
    
    def evaluate(self):
       device = "mps" if (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()) else ("cuda" if torch.cuda.is_available() else "cpu")
       tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path,use_fast=False)
       model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       dataset = load_from_disk(self.config.data_path)["test"]
       print(self.config.tokenizer_path)


       rouge_name = "rouge1", "rouge2", "rougeL", "rougeLsum"
       metric = evaluate.load("rouge", rouge_types=rouge_name)
       score = self.calculate_metric_on_test_ds(dataset, metric, model, tokenizer, device=device)
       rouge_dict = {k: round(v*100, 4) for k,v in score.items()}

       df = pd.DataFrame(rouge_dict, index=[0])
       df.to_csv(self.config.matric_file_name, index=False)
       

In [ ]:
try:
  config = configurationManager()
  model_evoluation_config = config.get_model_evoluation_config()
  model_evoluation = ModelEvoluation(config=model_evoluation_config)
  model_evoluation.evaluate()
except Exception as e:
  raise e

[2026-06-16 11:22:53,171: INFO: >>> Trying to open: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/config/config.yaml]
[2026-06-16 11:22:53,175: INFO: YAML file: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/config/config.yaml loaded successfully]
[2026-06-16 11:22:53,175: INFO: >>> Trying to open: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/params.yaml]
[2026-06-16 11:22:53,177: INFO: YAML file: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/params.yaml loaded successfully]
[2026-06-16 11:22:53,177: INFO: Directory created at: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/artifacts]
[2026-06-16 11:22:53,177: INFO: Directory created at: /Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/artifacts/model_evaluation]


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

/Users/akashkumarsinha/Desktop/Text_summerization/Text_Summerization/artifacts/model_trainer/tokenizer


  0%|          | 0/52 [00:00<?, ?it/s]